In [ ]:
# Root of the shared project folder on Google Drive.
# Team members mounted it at different places in their Drives, so all paths
# below are built from this single constant. Change it to match your setup.
DRIVE_ROOT = "/content/drive/MyDrive/furniture-retrieval"

In [ ]:
# 🚀 CLIP Embedder v2 - Patch-Based Processing

**Özellikler:**
- 📦 Patch-based: Her klasörden 2K'lık parçalar halinde işler
- 💾 Drive'a anında kayıt: Her patch bitince CSV + NPY olarak kaydeder
- 🔄 Resume: Kesintide kaldığı yerden devam eder
- ⚖️ Dengeli ilerleme: Tüm klasörler paralel ilerler

**Yapı:**
```
/content/data/images/          <- Input (Colab disk)
├── folder1/
├── folder2/
└── ...

/content/drive/MyDrive/embeddings/  <- Output (Drive - kalıcı)
├── checkpoint.json
├── folder1_patch_0.npy
├── folder1_patch_0.csv
├── folder1_patch_1.npy
└── ...
```

In [ ]:
# 📦 Gerekli paketleri yükle
!pip install -q open-clip-torch pandas pyarrow tqdm

In [ ]:
# 🔗 Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ⚙️ AYARLAR
# ============================================

# Resimlerin bulunduğu klasör (Colab disk)
INPUT_DIR = "/content/data/images"

# Embedding'lerin kaydedileceği klasör (Drive - kalıcı)
OUTPUT_DIR = f"{DRIVE_ROOT}/embeddings"

# Patch boyutu (her seferde işlenecek resim sayısı)
PATCH_SIZE = 2000

# Batch boyutu (GPU için)
BATCH_SIZE = 128

# CLIP model
MODEL_NAME = "ViT-B-32"

# Output klasörünü oluştur
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📁 Input: {INPUT_DIR}")
print(f"📁 Output: {OUTPUT_DIR}")
print(f"📦 Patch size: {PATCH_SIZE}")
print(f"🔢 Batch size: {BATCH_SIZE}")

In [ ]:
# 📚 Import'lar
import os
import json
from pathlib import Path
from typing import List, Tuple, Dict, Optional, Set
from datetime import datetime

import torch
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import pandas as pd
import open_clip

# GPU kontrolü
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🎮 Device: {device}")
if device == "cuda":
    print(f"💾 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif', '.tiff'}
CHECKPOINT_FILE = "checkpoint.json"

In [ ]:
# 🔧 Checkpoint Fonksiyonları

def save_checkpoint(output_dir: Path, state: dict):
    """Checkpoint kaydet."""
    checkpoint_path = output_dir / CHECKPOINT_FILE
    state['last_updated'] = datetime.now().isoformat()
    with open(checkpoint_path, 'w') as f:
        json.dump(state, f, indent=2)


def load_checkpoint(output_dir: Path) -> dict:
    """Checkpoint yükle veya yeni oluştur."""
    checkpoint_path = output_dir / CHECKPOINT_FILE
    
    if checkpoint_path.exists():
        try:
            with open(checkpoint_path) as f:
                state = json.load(f)
            print(f"📂 Checkpoint bulundu: {state.get('last_updated', 'unknown')}")
            print(f"   İşlenen patch sayısı: {state.get('total_patches_done', 0)}")
            return state
        except:
            pass
    
    # Yeni checkpoint
    return {
        "folders": {},  # folder_name -> {"total_images": N, "patches_done": M, "current_index": X}
        "total_patches_done": 0,
        "total_images_done": 0
    }


def get_all_images_in_folder(folder: Path) -> List[Path]:
    """Klasördeki tüm resimleri bul."""
    images = []
    for ext in IMAGE_EXTENSIONS:
        images.extend(folder.glob(f'*{ext}'))
        images.extend(folder.glob(f'*{ext.upper()}'))
    return sorted(images, key=lambda x: x.name)  # Deterministik sıralama


print("✅ Checkpoint fonksiyonları yüklendi")

In [ ]:
# 🧠 CLIP Model Yükleme

def load_clip_model(model_name: str = "ViT-B-32"):
    """CLIP modelini yükle."""
    print(f"🧠 Model yükleniyor: {model_name}")
    model, _, preprocess = open_clip.create_model_and_transforms(
        model_name, 
        pretrained='laion2b_s34b_b79k'
    )
    model = model.to(device).eval()
    print("✅ Model yüklendi")
    return model, preprocess


def process_batch(model, preprocess, image_paths: List[Path], batch_size: int = 128) -> Tuple[List[str], np.ndarray]:
    """Bir batch resmi işle ve embedding çıkar."""
    valid_paths = []
    all_embeddings = []
    
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        batch_images = []
        batch_valid_paths = []
        
        for path in batch_paths:
            try:
                img = Image.open(path).convert('RGB')
                img = preprocess(img)
                batch_images.append(img)
                batch_valid_paths.append(str(path))
            except Exception as e:
                continue
        
        if not batch_images:
            continue
        
        with torch.no_grad():
            images_tensor = torch.stack(batch_images).to(device)
            features = model.encode_image(images_tensor)
            features = features / features.norm(dim=-1, keepdim=True)
            features = features.cpu().numpy()
        
        all_embeddings.append(features)
        valid_paths.extend(batch_valid_paths)
    
    if all_embeddings:
        combined = np.vstack(all_embeddings)
    else:
        combined = np.array([])
    
    return valid_paths, combined


print("✅ CLIP fonksiyonları yüklendi")

In [ ]:
# 🚀 Ana İşlem Fonksiyonu - Patch-Based Round Robin

def run_patch_embedder(
    input_dir: str,
    output_dir: str,
    patch_size: int = 2000,
    batch_size: int = 128,
    model_name: str = "ViT-B-32"
):
    """
    Tüm klasörleri patch-patch dolaşarak embedding çıkar.
    Round-robin: Her klasörden sırayla 1 patch al, dengeli ilerle.
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Model yükle
    model, preprocess = load_clip_model(model_name)
    
    # Klasörleri bul
    folders = sorted([d for d in input_path.iterdir() if d.is_dir()], key=lambda x: x.name)
    print(f"📁 Bulunan klasör: {len(folders)}")
    
    # Checkpoint yükle
    state = load_checkpoint(output_path)
    
    # Her klasör için resim listesini hazırla
    folder_info = {}
    for folder in folders:
        folder_name = folder.name
        images = get_all_images_in_folder(folder)
        
        # Checkpoint'ten devam bilgisini al
        if folder_name in state["folders"]:
            current_index = state["folders"][folder_name]["current_index"]
        else:
            current_index = 0
            state["folders"][folder_name] = {
                "total_images": len(images),
                "patches_done": 0,
                "current_index": 0
            }
        
        folder_info[folder_name] = {
            "path": folder,
            "images": images,
            "current_index": current_index,
            "total": len(images)
        }
        
        remaining = len(images) - current_index
        print(f"   📂 {folder_name}: {len(images)} resim ({remaining} kalan)")
    
    # Round-robin döngüsü
    total_remaining = sum(info["total"] - info["current_index"] for info in folder_info.values())
    print(f"\n🎯 Toplam işlenecek: {total_remaining} resim")
    
    pbar = tqdm(total=total_remaining, desc="🚀 Toplam İlerleme")
    
    while True:
        # Herhangi bir klasörde işlenecek resim var mı?
        active_folders = [
            name for name, info in folder_info.items() 
            if info["current_index"] < info["total"]
        ]
        
        if not active_folders:
            break
        
        # Her aktif klasörden sırayla 1 patch al
        for folder_name in active_folders:
            info = folder_info[folder_name]
            images = info["images"]
            start_idx = info["current_index"]
            end_idx = min(start_idx + patch_size, info["total"])
            
            if start_idx >= info["total"]:
                continue
            
            patch_images = images[start_idx:end_idx]
            patch_num = state["folders"][folder_name]["patches_done"]
            
            print(f"\n📦 {folder_name} - Patch {patch_num} ({start_idx}-{end_idx})")
            
            # Embedding çıkar
            paths, embeddings = process_batch(model, preprocess, patch_images, batch_size)
            
            if len(paths) > 0:
                # CSV kaydet (image paths)
                csv_file = output_path / f"{folder_name}_patch_{patch_num}.csv"
                df = pd.DataFrame({
                    "image_path": paths,
                    "folder": folder_name
                })
                df.to_csv(csv_file, index=False)
                
                # NPY kaydet (embeddings)
                npy_file = output_path / f"{folder_name}_patch_{patch_num}.npy"
                np.save(npy_file, embeddings)
                
                print(f"   ✅ Kaydedildi: {len(paths)} resim -> {csv_file.name}")
            
            # State güncelle
            info["current_index"] = end_idx
            state["folders"][folder_name]["current_index"] = end_idx
            state["folders"][folder_name]["patches_done"] += 1
            state["total_patches_done"] += 1
            state["total_images_done"] += len(paths)
            
            # Checkpoint kaydet
            save_checkpoint(output_path, state)
            
            # Progress güncelle
            pbar.update(len(patch_images))
            
            # GPU memory temizle
            torch.cuda.empty_cache()
    
    pbar.close()
    
    # Final özet
    print(f"\n{'='*50}")
    print("✅ TAMAMLANDI!")
    print(f"📊 Toplam patch: {state['total_patches_done']}")
    print(f"📊 Toplam resim: {state['total_images_done']}")
    print(f"📁 Çıktı: {output_path}")
    
    return state


print("✅ Ana işlem fonksiyonu yüklendi")

---
## ▶️ Çalıştır

Aşağıdaki hücreyi çalıştır. Kesinti olursa tekrar çalıştır - kaldığı yerden devam eder.

In [ ]:
# ▶️ ÇALIŞTIR
state = run_patch_embedder(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    patch_size=PATCH_SIZE,
    batch_size=BATCH_SIZE,
    model_name=MODEL_NAME
)

---
## 📊 Sonuçları İncele

In [ ]:
# 📊 Checkpoint durumunu görüntüle
output_path = Path(OUTPUT_DIR)
checkpoint_path = output_path / CHECKPOINT_FILE

if checkpoint_path.exists():
    with open(checkpoint_path) as f:
        state = json.load(f)
    
    print(f"📊 DURUM")
    print(f"{'='*50}")
    print(f"Son güncelleme: {state.get('last_updated', 'unknown')}")
    print(f"Toplam patch: {state.get('total_patches_done', 0)}")
    print(f"Toplam resim: {state.get('total_images_done', 0)}")
    print(f"\n📁 Klasör durumları:")
    
    for folder_name, info in sorted(state.get('folders', {}).items()):
        total = info.get('total_images', 0)
        done = info.get('current_index', 0)
        patches = info.get('patches_done', 0)
        pct = (done / total * 100) if total > 0 else 0
        status = "✅" if done >= total else "🔄"
        print(f"   {status} {folder_name}: {done}/{total} ({pct:.1f}%) - {patches} patch")
else:
    print("⚠️ Checkpoint bulunamadı. Henüz çalıştırılmamış.")

In [ ]:
# 📁 Oluşturulan dosyaları listele
output_path = Path(OUTPUT_DIR)

csv_files = sorted(output_path.glob("*.csv"))
npy_files = sorted(output_path.glob("*.npy"))

print(f"📊 Drive'daki dosyalar:")
print(f"   CSV: {len(csv_files)}")
print(f"   NPY: {len(npy_files)}")

# Toplam boyut
total_size = sum(f.stat().st_size for f in output_path.iterdir() if f.is_file())
print(f"   Toplam boyut: {total_size / 1e6:.1f} MB")

print(f"\n📁 Son 10 dosya:")
for f in sorted(output_path.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True)[:10]:
    if f.is_file() and f.name != CHECKPOINT_FILE:
        print(f"   {f.name} ({f.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# 🔍 Örnek embedding kontrol et
output_path = Path(OUTPUT_DIR)

npy_files = list(output_path.glob("*.npy"))
if npy_files:
    sample_file = npy_files[0]
    embeddings = np.load(sample_file)
    
    csv_file = sample_file.with_suffix('.csv')
    df = pd.read_csv(csv_file)
    
    print(f"📁 Örnek: {sample_file.name}")
    print(f"   Embedding shape: {embeddings.shape}")
    print(f"   CSV rows: {len(df)}")
    print(f"\n📊 İlk 3 resim:")
    for i, row in df.head(3).iterrows():
        print(f"   {Path(row['image_path']).name}")
    print(f"\n📊 Embedding örneği (ilk 10 değer):")
    print(f"   {embeddings[0][:10]}")
else:
    print("⚠️ Henüz embedding dosyası yok.")

---
## 🔗 Patch'leri Birleştir

Tüm işlem bittikten sonra, dataset_creator için patch'leri tek dosyada birleştir.

In [ ]:
# 🔗 Tüm patch'leri birleştir (dataset_creator için)

def merge_all_patches(output_dir: str):
    """
    Tüm patch CSV ve NPY dosyalarını birleştir.
    Çıktı:
      - all_embeddings.npy: (N, 512) tüm embedding'ler
      - all_metadata.parquet: image_path, folder bilgisi
      - summary.json: dataset_creator uyumlu özet
    """
    output_path = Path(output_dir)
    
    # Tüm CSV ve NPY dosyalarını bul
    csv_files = sorted(output_path.glob("*_patch_*.csv"))
    
    if not csv_files:
        print("⚠️ Patch dosyası bulunamadı!")
        return
    
    print(f"📦 {len(csv_files)} patch dosyası bulundu")
    
    all_dfs = []
    all_embeddings = []
    
    for csv_file in tqdm(csv_files, desc="Patch'ler birleştiriliyor"):
        npy_file = csv_file.with_suffix('.npy')
        
        if not npy_file.exists():
            print(f"⚠️ NPY bulunamadı: {npy_file.name}")
            continue
        
        # CSV yükle
        df = pd.read_csv(csv_file)
        all_dfs.append(df)
        
        # NPY yükle
        embeddings = np.load(npy_file)
        all_embeddings.append(embeddings)
    
    # Birleştir
    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_embeddings = np.vstack(all_embeddings)
    
    print(f"\n📊 Birleştirme sonucu:")
    print(f"   Toplam resim: {len(combined_df)}")
    print(f"   Embedding shape: {combined_embeddings.shape}")
    
    # Kaydet
    # 1. Embeddings (tek NPY)
    embeddings_file = output_path / "all_embeddings.npy"
    np.save(embeddings_file, combined_embeddings)
    print(f"   ✅ {embeddings_file.name} ({embeddings_file.stat().st_size / 1e6:.1f} MB)")
    
    # 2. Metadata (Parquet - dataset_creator uyumlu)
    metadata_file = output_path / "all_metadata.parquet"
    combined_df.to_parquet(metadata_file, index=False)
    print(f"   ✅ {metadata_file.name}")
    
    # 3. Summary (dataset_creator uyumlu format)
    folder_counts = combined_df['folder'].value_counts().to_dict()
    summary = {
        "folders": {
            folder: {
                "count": count,
                "embeddings_file": str(embeddings_file),
                "metadata_file": str(metadata_file)
            }
            for folder, count in folder_counts.items()
        },
        "total_images": len(combined_df),
        "embedding_dim": combined_embeddings.shape[1]
    }
    
    summary_file = output_path / "summary.json"
    with open(summary_file, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"   ✅ {summary_file.name}")
    
    # Klasör dağılımı
    print(f"\n📁 Klasör dağılımı:")
    for folder, count in sorted(folder_counts.items()):
        print(f"   {folder}: {count}")
    
    return combined_df, combined_embeddings


print("✅ Merge fonksiyonu yüklendi")

In [ ]:
# 🔗 BİRLEŞTİR - Tüm embedding işlemi bittikten sonra çalıştır
metadata, embeddings = merge_all_patches(OUTPUT_DIR)

In [ ]:
# 🔍 Birleştirilmiş dosyaları kontrol et
output_path = Path(OUTPUT_DIR)

print("📁 Dataset Creator için hazır dosyalar:")
print(f"{'='*50}")

files_to_check = ["all_embeddings.npy", "all_metadata.parquet", "summary.json"]
for fname in files_to_check:
    fpath = output_path / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / 1e6
        print(f"   ✅ {fname} ({size_mb:.1f} MB)")
    else:
        print(f"   ❌ {fname} (bulunamadı)")

print(f"\n💡 Dataset Creator kullanımı:")
print(f"   python dataset_creator.py -e {OUTPUT_DIR} -o ./golden_dataset --method greedy --threshold 0.92")